# Live Coding: Clustering avanzado y reglas de asociación en retail

Dataset: **Online Retail / Online Retail II** (UCI / Kaggle).


## 1. Carga y preparación de datos

En esta sección:
1. Cargamos el dataset de retail desde un CSV.
2. Limpiamos datos nulos y devoluciones.
3. Nos quedamos con un país para simplificar.
4. Construimos variables numéricas por cliente (NumInvoices, NumItems, TotalSpent).
5. Estandarizamos las variables para poder aplicar clustering.


In [ ]:
import pandas as pd
import numpy as np

# Mostrar todas las columnas sin truncar
display_options = dict(max_columns=None, width=1000)
for k, v in display_options.items():
    pd.set_option(k, v)

# 1.1 Carga del dataset (ajusta la ruta a tu archivo CSV)
# Por ejemplo: 'online_retail_II.csv' descargado de Kaggle/UCI
file_path = r"C:\Users\Yuri\Proyectos\demo\data\online_retail_II.csv"
  # <-- CAMBIA ESTA RUTA SI ES NECESARIO

df = pd.read_csv(file_path, encoding="ISO-8859-1")
print("Shape original:", df.shape)
df.head()

In [ ]:
# 1.2 Limpieza básica
# - Quitamos filas sin descripción o sin Customer ID
# - Quitamos devoluciones (Invoice que empieza con 'C')

df = df.dropna(subset=["Description", "Customer ID"])
df = df[~df["Invoice"].astype(str).str.startswith("C")]
print("Shape tras limpiar nulos y devoluciones:", df.shape)

df.head()

In [ ]:
# 1.3 Nos quedamos con un solo país para simplificar el análisis

print("Países disponibles:", df["Country"].unique())

country_filter = "United Kingdom"  # puedes cambiarlo en vivo si quieres

df = df[df["Country"] == country_filter]
print("Shape tras filtrar país:", df.shape)

df.head()

In [ ]:
# 1.4 Creamos una variable de importe total por línea

df["TotalPrice"] = df["Quantity"] * df["Price"]

# 1.5 Agregamos por cliente (Customer ID)
customer_df = (
    df.groupby("Customer ID")
      .agg(
          NumInvoices=("Invoice", "nunique"),
          NumItems=("Quantity", "sum"),
          TotalSpent=("TotalPrice", "sum")
      )
      .reset_index()
)

print("Clientes únicos:", customer_df.shape[0])
customer_df.head()

In [ ]:
# 1.6 Estandarizamos las variables numéricas para clustering
from sklearn.preprocessing import StandardScaler

features = ["NumInvoices", "NumItems", "TotalSpent"]

scaler = StandardScaler()
X = scaler.fit_transform(customer_df[features])

print("Shape de la matriz X para clustering:", X.shape)

## 2. Aplicación de DBSCAN

En esta sección:
1. Usamos una gráfica k-dist para elegir un valor razonable de `eps`.
2. Aplicamos DBSCAN sobre las variables estandarizadas.
3. Exploramos cuántos clusters y outliers detecta.


In [ ]:
from sklearn.cluster import DBSCAN
from sklearn.neighbors import NearestNeighbors
import matplotlib.pyplot as plt

# 2.1 Curva k-dist para ayudar a elegir eps
neighbors = NearestNeighbors(n_neighbors=5)
neighbors_fit = neighbors.fit(X)

# distances: matriz de distancias; nos quedamos con la distancia al 5º vecino
distances, indices = neighbors_fit.kneighbors(X)
distances = np.sort(distances[:, -1])

plt.figure(figsize=(6, 4))
plt.plot(distances)
plt.ylabel("Distancia al 5º vecino")
plt.xlabel("Puntos ordenados")
plt.title("Curva k-dist para seleccionar eps")
plt.grid(True)
plt.show()

In [ ]:
# 2.2 Aplicamos DBSCAN
# Puedes ajustar 'eps' y 'min_samples' en vivo según la curva anterior

eps = 0.8        # valor inicial de ejemplo
min_samples = 10 # número mínimo de puntos en un vecindario denso

dbscan = DBSCAN(eps=eps, min_samples=min_samples)
labels_dbscan = dbscan.fit_predict(X)

customer_df["DBSCAN_cluster"] = labels_dbscan

unique, counts = np.unique(labels_dbscan, return_counts=True)
cluster_info = dict(zip(unique, counts))
print("Tamaño de cada cluster (incluyendo -1 como ruido):")
cluster_info

## 3. Visualización 2D con t-SNE

En esta sección:
1. Reducimos la dimensión de X a 2 usando t-SNE.
2. Visualizamos a los clientes coloreados según el cluster DBSCAN.


In [ ]:
from sklearn.manifold import TSNE

# El parámetro 'perplexity' suele funcionar bien entre 5 y 50

tsne = TSNE(n_components=2, random_state=42, perplexity=30)
X_tsne = tsne.fit_transform(X)

plt.figure(figsize=(7, 6))
scatter = plt.scatter(
    X_tsne[:, 0],
    X_tsne[:, 1],
    c=customer_df["DBSCAN_cluster"],
    cmap="tab10",
    s=15,
    alpha=0.8
)
plt.colorbar(scatter, label="Cluster DBSCAN")
plt.title("Clientes embebidos con t-SNE coloreados por DBSCAN")
plt.xlabel("t-SNE 1")
plt.ylabel("t-SNE 2")
plt.grid(True)
plt.show()

## 4. Agrupamiento jerárquico y dendrogramas

En esta sección:
1. Muestreamos una parte de los clientes para hacer el dendrograma.
2. Construimos un dendrograma con enlace `ward`.
3. Creamos clusters jerárquicos con `AgglomerativeClustering`.


In [ ]:
from sklearn.cluster import AgglomerativeClustering
from scipy.cluster.hierarchy import dendrogram, linkage

# 4.1 Dendrograma con una muestra para que sea más rápido
sample_size = min(300, X.shape[0])
np.random.seed(42)
sample_idx = np.random.choice(len(X), size=sample_size, replace=False)
X_sample = X[sample_idx]

Z = linkage(X_sample, method="ward")

plt.figure(figsize=(10, 4))
dendrogram(
    Z,
    truncate_mode="lastp",  # muestra solo las últimas fusiones
    p=20,
    leaf_rotation=45.,
    leaf_font_size=8.
)
plt.title("Dendrograma truncado (ward)")
plt.xlabel("Clusters")
plt.ylabel("Distancia")
plt.show()

In [ ]:
# 4.2 Clustering aglomerativo con un número fijo de clusters

n_clusters = 4  # puedes cambiar este número en vivo

agg = AgglomerativeClustering(n_clusters=n_clusters, linkage="ward")
labels_hier = agg.fit_predict(X)

customer_df["HIER_cluster"] = labels_hier

unique_h, counts_h = np.unique(labels_hier, return_counts=True)
cluster_info_h = dict(zip(unique_h, counts_h))
print("Tamaño de cada cluster jerárquico:")
cluster_info_h

## 5. Aplicación de Apriori (mlxtend)

En esta sección:
1. Convertimos el dataset en una lista de transacciones (facturas).
2. Hacemos un one-hot encoding de los productos.
3. Extraemos itemsets frecuentes con el algoritmo Apriori.


In [ ]:
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

# 5.1 Lista de transacciones: cada Invoice es una lista de descripciones
basket_series = (
    df.groupby("Invoice")["Description"]
      .apply(list)
)

transactions = basket_series.tolist()

print("Número de transacciones:", len(transactions))
transactions[:3]

In [ ]:
# 5.2 One-hot encoding de las transacciones

te = TransactionEncoder()
te_array = te.fit(transactions).transform(transactions)

basket_df = pd.DataFrame(te_array, columns=te.columns_)
print("Shape de la matriz one-hot:", basket_df.shape)

basket_df.iloc[:5, :10]  # mostramos solo las primeras columnas

In [ ]:
# 5.3 Itemsets frecuentes con Apriori

min_support = 0.02  # 2% de las transacciones; puedes ajustar en vivo

frequent_itemsets = apriori(
    basket_df,
    min_support=min_support,
    use_colnames=True
)

frequent_itemsets = frequent_itemsets.sort_values("support", ascending=False)
print("Número de itemsets frecuentes:", frequent_itemsets.shape[0])
frequent_itemsets.head(10)

## 6. Filtrado de reglas significativas

En esta sección:
1. Generamos reglas de asociación a partir de los itemsets frecuentes.
2. Filtramos por soporte, confianza y lift.
3. Inspeccionamos las reglas más interesantes.


In [ ]:
# 6.1 Generación de reglas de asociación

rules = association_rules(
    frequent_itemsets,
    metric="confidence",
    min_threshold=0.3
)

print("Número de reglas generadas (sin filtro extra):", rules.shape[0])
rules.head()

In [ ]:
# 6.2 Filtrado de reglas significativas

support_threshold = 0.02
confidence_threshold = 0.4
lift_threshold = 1.2

rules_filtered = rules[
    (rules["support"] >= support_threshold) &
    (rules["confidence"] >= confidence_threshold) &
    (rules["lift"] >= lift_threshold)
].sort_values("lift", ascending=False)

print("Reglas tras el filtrado:", rules_filtered.shape[0])

rules_filtered[["antecedents", "consequents", "support", "confidence", "lift"]].head(10)

## 7. Visualización de soporte / confianza / lift

En esta sección:
1. Creamos un scatterplot donde cada punto es una regla.
2. El eje X es el soporte, el eje Y la confianza y el color representa el lift.


In [ ]:
plt.figure(figsize=(7, 6))
scatter = plt.scatter(
    rules_filtered["support"],
    rules_filtered["confidence"],
    c=rules_filtered["lift"],
    cmap="viridis",
    s=40,
    alpha=0.7
)
plt.colorbar(scatter, label="Lift")
plt.xlabel("Soporte")
plt.ylabel("Confianza")
plt.title("Reglas de asociación: soporte vs confianza (color = lift)")
plt.grid(True)
plt.show()